In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json
import os

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'Doc': '#4C72B0', 'Img': '#DD8452', 'Movie': '#55A868',
    'Rec': '#C44E52', 'BGM': '#8172B3'
}
DPI = 300

In [ ]:
# fig08_domain_hitrate_comparison.png - 5-domain hit rate comparison

domains = ['Doc', 'Img', 'Movie', 'Rec', 'BGM']
configs = ['Dense only', 'Dense+Sparse', 'Dense+Sparse+ASF']

data = {
    'Doc':   [0.60, 0.60, 0.53],
    'Img':   [0.50, 0.50, 0.50],
    'Movie': [0.33, 0.33, 0.33],
    'Rec':   [0.40, 0.40, 0.37],
    'BGM':   [0.35, 0.35, 0.35],
}

fig, ax = plt.subplots(figsize=(14, 7))

n_domains = len(domains)
n_configs = len(configs)
bar_width = 0.22
group_gap = 0.1
x = np.arange(n_domains) * (n_configs * bar_width + group_gap)

shade_factors = [1.0, 0.75, 0.50]

for ci, config in enumerate(configs):
    heights = [data[d][ci] for d in domains]
    bar_colors = []
    for d in domains:
        base = tuple(int(COLORS[d].lstrip('#')[i:i+2], 16) / 255 for i in (0, 2, 4))
        f = shade_factors[ci]
        bar_colors.append((base[0]*f, base[1]*f, base[2]*f))
    offset = (ci - (n_configs - 1) / 2) * bar_width
    bars = ax.bar(x + offset, heights, bar_width, label=config,
                  color=bar_colors, edgecolor='white', linewidth=0.5)
    for bar, h in zip(bars, heights):
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                f'{h:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(domains, fontsize=13)
ax.set_ylabel('Hit Rate', fontsize=12)
ax.set_ylim(0, 0.75)
ax.set_title('도메인별 Hit Rate 비교 (Dense / Sparse / ASF)', fontsize=15, fontweight='bold', pad=14)
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)

legend_patches = [
    mpatches.Patch(color='#555555', label='Dense only'),
    mpatches.Patch(color='#999999', label='Dense+Sparse'),
    mpatches.Patch(color='#cccccc', label='Dense+Sparse+ASF'),
]
ax.legend(handles=legend_patches, fontsize=11, loc='upper right')

plt.tight_layout()
out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig08_domain_hitrate_comparison.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# fig08_di_vs_mr_comparison.png - DI_TriCHEF vs MR_TriCHEF side-by-side infographic

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 8))

def draw_infographic_panel(ax, title, color, domains_list,
                            key_metrics, components, cv_auc_dict):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')

    # Header box
    header_rect = mpatches.FancyBboxPatch((0.3, 8.5), 9.4, 1.2,
        boxstyle='round,pad=0.15', linewidth=2,
        edgecolor=color, facecolor=color, alpha=0.85, zorder=2)
    ax.add_patch(header_rect)
    ax.text(5, 9.15, title, ha='center', va='center',
            fontsize=16, fontweight='bold', color='white', zorder=3)

    # Domains label
    ax.text(0.5, 7.95, '대상 도메인', fontsize=11, fontweight='bold', color='#333333')
    domain_colors_map = {'Doc': '#4C72B0', 'Img': '#DD8452',
                         'Movie': '#55A868', 'Rec': '#C44E52', 'BGM': '#8172B3'}
    for idx, dom in enumerate(domains_list):
        dc = domain_colors_map.get(dom, '#888888')
        drect = mpatches.FancyBboxPatch((0.5 + idx * 2.5, 7.35), 2.0, 0.55,
            boxstyle='round,pad=0.1', linewidth=1.5,
            edgecolor=dc, facecolor=dc, alpha=0.8, zorder=2)
        ax.add_patch(drect)
        ax.text(1.5 + idx * 2.5, 7.625, dom, ha='center', va='center',
                fontsize=12, fontweight='bold', color='white', zorder=3)

    # Key metrics box
    ax.text(0.5, 6.85, '핵심 성능 지표', fontsize=11, fontweight='bold', color='#333333')
    metrics_rect = mpatches.FancyBboxPatch((0.3, 5.5), 9.4, 1.25,
        boxstyle='round,pad=0.15', linewidth=1.5,
        edgecolor=color, facecolor=color, alpha=0.12, zorder=1)
    ax.add_patch(metrics_rect)
    for mi, metric in enumerate(key_metrics):
        ax.text(0.7 + mi * 4.7, 6.12, metric, ha='left', va='center',
                fontsize=10.5, color='#1a1a1a', zorder=2)

    # Components box
    ax.text(0.5, 5.05, '구성 컴포넌트', fontsize=11, fontweight='bold', color='#333333')
    for ci, comp in enumerate(components):
        crect = mpatches.FancyBboxPatch((0.4 + ci * 3.15, 4.3), 2.7, 0.6,
            boxstyle='round,pad=0.1', linewidth=1.2,
            edgecolor=color, facecolor='white', alpha=1.0, zorder=2)
        ax.add_patch(crect)
        ax.text(1.75 + ci * 3.15, 4.6, comp, ha='center', va='center',
                fontsize=9.5, color=color, fontweight='bold', zorder=3)

    # CV AUC section
    ax.text(0.5, 3.85, 'MPLC CV AUC', fontsize=11, fontweight='bold', color='#333333')
    bar_y_base = 1.2
    bar_h = 0.5
    auc_items = list(cv_auc_dict.items())
    max_auc = 1.0
    bar_total_w = 9.0
    for ai, (label, auc_val) in enumerate(auc_items):
        y_pos = bar_y_base + ai * 0.75
        dc = domain_colors_map.get(label, color)
        fill_w = (auc_val / max_auc) * bar_total_w
        bg_rect = mpatches.FancyBboxPatch((0.5, y_pos), bar_total_w, bar_h,
            boxstyle='round,pad=0.05', linewidth=0.8,
            edgecolor='#cccccc', facecolor='#f0f0f0', zorder=1)
        ax.add_patch(bg_rect)
        fill_rect = mpatches.FancyBboxPatch((0.5, y_pos), fill_w, bar_h,
            boxstyle='round,pad=0.05', linewidth=0,
            edgecolor='none', facecolor=dc, alpha=0.75, zorder=2)
        ax.add_patch(fill_rect)
        ax.text(0.7, y_pos + bar_h / 2, label, ha='left', va='center',
                fontsize=9, fontweight='bold', color='white', zorder=3)
        ax.text(0.5 + fill_w + 0.1, y_pos + bar_h / 2, f'{auc_val:.3f}',
                ha='left', va='center', fontsize=9, color='#333333', zorder=3)

# Left panel: DI_TriCHEF
draw_infographic_panel(
    ax_left,
    title='DI_TriCHEF',
    color='#4C72B0',
    domains_list=['Doc', 'Img'],
    key_metrics=['LOO R@1 = 83.3%\n(best α=0.2)', 'MRR = 0.869'],
    components=['Z-axis (DINOv2)', 'BLIP Captioner', 'MPLC'],
    cv_auc_dict={'Doc': 0.985, 'Img': 0.921}
)

# Right panel: MR_TriCHEF
draw_infographic_panel(
    ax_right,
    title='MR_TriCHEF',
    color='#55A868',
    domains_list=['Movie', 'Rec', 'BGM'],
    key_metrics=['Movie Hit@5 = 96.7%', 'Rec Hit@5 = 100%'],
    components=['STT (Whisper)', 'LangGraph', 'Segmentation'],
    cv_auc_dict={'Video': 0.986, 'Audio': 0.989, 'BGM': 0.922}
)

fig.suptitle('DI_TriCHEF vs MR_TriCHEF 파이프라인 비교',
             fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig08_di_vs_mr_comparison.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# fig08_latency_profile.png - Latency profile bar chart with error bars

domains_lat = ['Doc', 'Img', 'Movie', 'Rec']
p50 = [72.5, 35.0, 46.0, 39.5]
p95 = [77.9, 38.1, 51.1, 45.4]
errors_upper = [p95[i] - p50[i] for i in range(len(p50))]

bar_colors = [COLORS[d] for d in domains_lat]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(domains_lat))
bars = ax.bar(x, p50, width=0.5, color=bar_colors, alpha=0.85,
              edgecolor='white', linewidth=0.8,
              yerr=[np.zeros(len(p50)), errors_upper],
              error_kw=dict(ecolor='#333333', capsize=8, capthick=2, elinewidth=2),
              label='p50 latency')

for bar, val, p95v in zip(bars, p50, p95):
    ax.text(bar.get_x() + bar.get_width() / 2, val / 2,
            f'p50\n{val}ms', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')
    ax.text(bar.get_x() + bar.get_width() / 2, p95v + 1.5,
            f'p95={p95v}ms', ha='center', va='bottom',
            fontsize=8.5, color='#333333')

ax.axhline(y=100, color='#E74C3C', linestyle='--', linewidth=1.8,
           label='Target (100ms)', alpha=0.9)
ax.text(len(domains_lat) - 0.5 + 0.3, 101.5, 'Target: 100ms',
        color='#E74C3C', fontsize=9, va='bottom')

ax.set_xticks(x)
ax.set_xticklabels(domains_lat, fontsize=13)
ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_ylim(0, 120)
ax.set_title('도메인별 검색 지연시간 프로파일 (ms)', fontsize=14, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
ax.legend(fontsize=10, loc='upper right')

plt.tight_layout()
out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig08_latency_profile.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# fig08_final_dashboard.png - 2x2 comprehensive dashboard

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ------- (0,0): Domain accuracy bar chart -------
ax00 = axes[0, 0]
domains_acc = ['Doc', 'Img', 'Movie', 'Rec', 'BGM']
metrics_labels = ['R@1', 'Hit@5', 'Hit@5', 'Hit@5', 'Conf']
acc_values = [0.833, 1.0, 0.967, 1.0, 0.76]
colors_acc = [COLORS[d] for d in domains_acc]

x00 = np.arange(len(domains_acc))
bars00 = ax00.bar(x00, acc_values, width=0.55, color=colors_acc,
                  alpha=0.88, edgecolor='white', linewidth=0.8)
for bar, val, ml in zip(bars00, acc_values, metrics_labels):
    ax00.text(bar.get_x() + bar.get_width() / 2, val + 0.008,
              f'{val:.3f}\n({ml})', ha='center', va='bottom',
              fontsize=8, fontweight='bold', color='#222222')
ax00.set_xticks(x00)
ax00.set_xticklabels(domains_acc, fontsize=11)
ax00.set_ylim(0, 1.15)
ax00.set_ylabel('Score', fontsize=10)
ax00.set_title('도메인별 최고 성능 지표', fontsize=12, fontweight='bold')
ax00.yaxis.grid(True, linestyle='--', alpha=0.5)
ax00.set_axisbelow(True)
ax00.axhline(1.0, color='gray', linestyle=':', linewidth=1, alpha=0.7)

# ------- (0,1): MPLC CV AUC horizontal bars -------
ax01 = axes[0, 1]
auc_labels = ['Doc', 'Img', 'Video', 'Audio', 'BGM']
auc_values = [0.985, 0.921, 0.986, 0.989, 0.922]
auc_colors = [COLORS.get(l, '#8172B3') for l in ['Doc', 'Img', 'Movie', 'Movie', 'BGM']]

y01 = np.arange(len(auc_labels))
hbars = ax01.barh(y01, auc_values, height=0.5, color=auc_colors,
                  alpha=0.88, edgecolor='white', linewidth=0.8)
for bar, val in zip(hbars, auc_values):
    ax01.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
              f'{val:.3f}', va='center', ha='left', fontsize=9, fontweight='bold')
ax01.set_yticks(y01)
ax01.set_yticklabels(auc_labels, fontsize=11)
ax01.set_xlim(0.85, 1.02)
ax01.set_xlabel('CV AUC', fontsize=10)
ax01.set_title('MPLC CV AUC (도메인별)', fontsize=12, fontweight='bold')
ax01.xaxis.grid(True, linestyle='--', alpha=0.5)
ax01.set_axisbelow(True)
ax01.axvline(1.0, color='gray', linestyle=':', linewidth=1, alpha=0.7)

# ------- (1,0): Latency profile (compact) -------
ax10 = axes[1, 0]
domains_lat2 = ['Doc', 'Img', 'Movie', 'Rec']
p50_2 = [72.5, 35.0, 46.0, 39.5]
p95_2 = [77.9, 38.1, 51.1, 45.4]
errors_upper2 = [p95_2[i] - p50_2[i] for i in range(len(p50_2))]
bar_colors_2 = [COLORS[d] for d in domains_lat2]

x10 = np.arange(len(domains_lat2))
ax10.bar(x10, p50_2, width=0.5, color=bar_colors_2, alpha=0.85,
         edgecolor='white', linewidth=0.8,
         yerr=[np.zeros(len(p50_2)), errors_upper2],
         error_kw=dict(ecolor='#333333', capsize=7, capthick=1.5, elinewidth=1.5))
ax10.axhline(y=100, color='#E74C3C', linestyle='--', linewidth=1.5, alpha=0.9)
ax10.text(len(domains_lat2) - 1 + 0.35, 101, '100ms target',
          color='#E74C3C', fontsize=8, va='bottom')
for xi, (val, p95v) in enumerate(zip(p50_2, p95_2)):
    ax10.text(xi, val / 2, f'{val}ms', ha='center', va='center',
              fontsize=8.5, fontweight='bold', color='white')
ax10.set_xticks(x10)
ax10.set_xticklabels(domains_lat2, fontsize=11)
ax10.set_ylabel('Latency (ms)', fontsize=10)
ax10.set_ylim(0, 120)
ax10.set_title('검색 지연시간 프로파일 (ms)', fontsize=12, fontweight='bold')
ax10.yaxis.grid(True, linestyle='--', alpha=0.5)
ax10.set_axisbelow(True)

# ------- (1,1): Calibration separation strength -------
ax11 = axes[1, 1]
sep_domains = ['Img', 'Doc', 'Movie', 'Rec', 'BGM']
sep_values = [0.8, 2.3, 3.2, 1.1, 3.65]
sep_units = ['°', '°', '°', '°', 'σ']
sep_colors = [COLORS[d] for d in sep_domains]

x11 = np.arange(len(sep_domains))
bars11 = ax11.bar(x11, sep_values, width=0.55, color=sep_colors,
                  alpha=0.88, edgecolor='white', linewidth=0.8)
for bar, val, unit in zip(bars11, sep_values, sep_units):
    ax11.text(bar.get_x() + bar.get_width() / 2, val + 0.05,
              f'{val}{unit}', ha='center', va='bottom',
              fontsize=9, fontweight='bold', color='#222222')
ax11.set_xticks(x11)
ax11.set_xticklabels(sep_domains, fontsize=11)
ax11.set_ylabel('Separation Strength', fontsize=10)
ax11.set_ylim(0, 4.5)
ax11.set_title('캘리브레이션 분리 강도', fontsize=12, fontweight='bold')
ax11.yaxis.grid(True, linestyle='--', alpha=0.5)
ax11.set_axisbelow(True)
ax11.text(0.98, 0.02, '단위: °(각도) / σ(표준편차)',
          transform=ax11.transAxes, ha='right', va='bottom',
          fontsize=8, color='gray', style='italic')

fig.suptitle('DB_insight 종합 성능 대시보드',
             fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig08_final_dashboard.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')